In [2]:
# Imports
import pickle
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.sparse import coo_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, auc, mean_squared_error
import xgboost as xgb

/home/luiz.gontijo/TCC_IC/OpenGraph/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
with open('node_classification/Resultados/embeddings/embedding0.pkl', 'rb') as f:
    embed = pickle.load(f)
embed

In [10]:
all_embeddings = []
for tensor_item in embed:
    all_embeddings.append(tensor_item.cpu().numpy())

concatenated_embeddings = np.vstack(all_embeddings)
embed_df = pd.DataFrame(concatenated_embeddings)

display(embed_df)
print(f"Shape of the full embeddings DataFrame: {embed_df.shape}")

,0,1,2,3,4,5,6,7,8,9,...,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023
0,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
1,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
2,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
3,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
4,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130351,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
130352,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
130353,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368
130354,-1.882011,-0.372567,-0.5958,-0.307821,0.225265,0.32181,-0.102871,0.046739,-0.031884,0.080006,...,-0.001991,-0.02158,-0.022678,0.025173,-0.052716,0.006261,-0.050028,-0.00305,0.005962,0.012368


Shape of the full embeddings DataFrame: (130356, 1024)


In [5]:
with open('node_classification/Resultados/embeddings/embedding0.pkl', 'rb') as f:
    embeddings = pickle.load(f)
embeddings

/home/luiz.gontijo/TCC_IC/OpenGraph/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()


[tensor([[ 1.1600e-01, -3.2892e-01, -2.3144e-01,  ..., -1.2334e-02,
           1.5017e-02,  1.5716e-02],
         [-1.5053e+00, -1.7112e-01,  2.0950e-01,  ...,  3.9491e-02,
          -5.5158e-02,  3.0014e-02],
         [ 6.9469e-01,  3.7398e-01, -5.0006e-02,  ...,  1.8930e-02,
          -1.0622e-02,  6.5731e-04],
         ...,
         [ 1.0544e-01, -9.8503e-02,  1.9388e-01,  ..., -2.7247e-02,
           1.4032e-02, -7.4767e-03],
         [ 2.9482e-01, -8.9186e-02, -1.0918e-01,  ...,  1.2743e-02,
           1.3642e-02, -7.0438e-03],
         [ 1.4621e+00, -3.0102e-01, -3.3926e-02,  ..., -9.4149e-03,
          -1.6507e-02, -6.2066e-03]], device='cuda:0'),
 tensor([[-0.1752,  0.2269,  0.2489,  ...,  0.0098,  0.0312, -0.0536],
         [ 0.1703,  0.1414,  0.0946,  ...,  0.0085, -0.0083, -0.0139],
         [ 0.7942, -0.1823, -0.0180,  ..., -0.0211,  0.0383, -0.0306],
         ...,
         [ 0.9533,  0.0438, -0.0414,  ...,  0.0011, -0.0071, -0.0098],
         [-0.0993,  0.3717, -0.0773,  .

In [4]:
!ls

Analise_SAML_D.ipynb  OpenGraph.ipynb  abregrafo.py	 link_prediction
History		      README.md        datasets		 node_classification
LICENSE		      Teste.ipynb      graph_generation  requirements.txt
Models		      __pycache__      imgs		 venv
